In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, ConfusionMatrixDisplay, recall_score, accuracy_score
from sklearn.metrics import  f1_score, mean_absolute_error, auc, roc_curve, confusion_matrix
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

Error: (1,15): error CS1002: ; expected
(1,15): error CS1525: Invalid expression term 'as'
(1,20): error CS1002: ; expected
(2,14): error CS1002: ; expected
(2,14): error CS1525: Invalid expression term 'as'
(2,19): error CS1002: ; expected
(3,24): error CS1003: Syntax error, '(' expected
(3,24): error CS1041: Identifier expected; 'as' is a keyword
(4,8): error CS1003: Syntax error, ',' expected
(4,26): error CS1001: Identifier expected
(4,26): error CS1003: Syntax error, ',' expected
(4,29): error CS1003: Syntax error, ',' expected
(5,8): error CS1003: Syntax error, ',' expected
(5,16): error CS1001: Identifier expected
(5,16): error CS1003: Syntax error, ',' expected
(5,19): error CS1003: Syntax error, ',' expected
(6,6): error CS1003: Syntax error, ',' expected
(6,29): error CS1003: Syntax error, ',' expected
(6,44): error CS1001: Identifier expected
(6,68): error CS1001: Identifier expected
(6,82): error CS1001: Identifier expected
(7,6): error CS1003: Syntax error, ',' expected
(7

In [46]:
data=pd.read_csv(r"Heart_Disease_Prediction.csv")
data.head()

In [47]:
data.info()

In [48]:
data.isnull().sum()

In [49]:
data.describe()

In [50]:
sns.boxplot(data['Cholesterol'])
outliers = data[(data['Cholesterol'] > 500)] 
outliers

In [51]:
data = data.drop(data[data.Cholesterol > 500].index)
sns.boxplot(data.Cholesterol)

In [52]:
data['Heart Disease'] = data['Heart Disease'].replace(['Presence', 'Absence'], [1,0])

In [54]:
data.info()

In [56]:
df_corr = data.corr()
plt.subplots(figsize=(20,15))
sns.heatmap(df_corr, annot = True)

In [64]:
# separate independent & dependent variables
X = data.iloc[:,0:14]  #independent columns
y = data.iloc[:,-1]    #target column i.e price range

# apply SelectKBest class to extract top 10 best features
bestfeatures = SelectKBest(score_func=chi2, k=10)
fit = bestfeatures.fit(X,y)
dfscores = pd.DataFrame(fit.scores_)
dfcolumns = pd.DataFrame(X.columns)

#concat two dataframes for better visualization 
featureScores = pd.concat([dfcolumns,dfscores],axis=1)
featureScores.columns = ['Specs','Score']  #naming the dataframe columns
print(featureScores.nlargest(14,'Score'))  #print 10 best features

In [65]:
featureScores = featureScores.sort_values(by='Score', ascending=False)
featureScores

In [66]:
features_list = featureScores["Specs"].tolist()[:13]
features_list

In [68]:
data = data[['Max HR',
 'Heart Disease',
 'Number of vessels fluro',
 'Thallium',
 'ST depression',
 'Cholesterol',
 'Exercise angina',
 'Age',
 'BP',
 'Chest pain type',
 'EKG results',
 'Sex',
 'Slope of ST']]

In [70]:
scaler = MinMaxScaler(feature_range=(0,1)) 

#assign scaler to column:
filtered_data = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)
filtered_data.describe()

In [71]:
data['Heart Disease'].value_counts()

In [72]:
P = data['Heart Disease']
X = data.drop(['Heart Disease'],axis = 1)

In [73]:
x_train, x_test, y_train, y_test = train_test_split(X,P, test_size = 0.25,random_state = 42)
y = y_test.values

In [74]:
result_table = pd.DataFrame(columns=['classifiers', 'fpr','tpr','auc'])

In [75]:
#Random Forest Classifier 
rf = RandomForestClassifier()
rf.fit(x_train, y_train)
test_res = rf.predict(x_test)


#Confusion Matrix 
confusion_matrix_test = confusion_matrix(y_test, test_res)

print("Confusion Matrix (Testing Data):")
obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###

fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

yproba = rf.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"RandomForestClassifier",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)


In [76]:
#Logistic Regression Classifier 
lr = LogisticRegression(max_iter = 1000)
lr.fit(x_train, y_train)
test_res = lr.predict(x_test)


#Confusion Matrix 
confusion_matrix_test = confusion_matrix(y_test, test_res)

print("Confusion Matrix (Testing Data):")
obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###

fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

######################################

yproba = lr.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"LogisticRegression",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)

In [77]:
#Simple Vector Classifier 
svc = SVC(kernel = 'linear', probability = True)
svc.fit(x_train, y_train)

test_res = svc.predict(x_test)

#Confusion Matrix 
confusion_matrix_test = confusion_matrix(y_test, test_res)

print("Confusion Matrix (Testing Data):")

obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###

fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

##################################

yproba = svc.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"SVC",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)

In [78]:
#Naive Bayes 
gnb = GaussianNB()
gnb.fit(x_train, y_train)

test_res = gnb.predict(x_test)

#Confusion Matrix 
confusion_matrix_test = confusion_matrix(y_test, test_res)

print("Confusion Matrix (Testing Data):")
obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###

fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

##########################################

yproba = gnb.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"NaiveBayes",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)

In [79]:
#Decision Tree Classsifier 

dtc = DecisionTreeClassifier(splitter = 'random')
dtc.fit(x_train, y_train)
test_res = dtc.predict(x_test)

#Confussion Matrix 
confusion_matrix_test = confusion_matrix(y_test, test_res)

print("Confusion Matrix (Testing Data):")
obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###

fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

##############################################

yproba = dtc.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"DecisionTreeClassifier",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)

In [80]:
#KN Classifier 
kn = KNeighborsClassifier(n_neighbors = 10)
kn.fit(x_train, y_train)

test_res = kn.predict(x_test)

#Confusion Matrix 

confusion_matrix_test = confusion_matrix(y_test, test_res)

obj = ConfusionMatrixDisplay(confusion_matrix_test)
obj.plot()
plt.show()

print("Accuracy Test : ", accuracy_score(y_test,test_res)*100)
print("Precision Score: ", precision_score(y_test, test_res)*100)
print("Recall Score : ", recall_score(y_test, test_res)*100)
print("F1 score : ", f1_score(y_test, test_res)*100)
print("Mean absolute error :", mean_absolute_error(y_test, test_res)*100)
###
fpr, tpr, thresholds  = roc_curve(y_test,test_res)
AUC = metrics.auc(fpr,tpr)
print("AUC score : ", AUC*100)

#####################################################


yproba = kn.predict_proba(x_test)
    
fpr, tpr, _ = roc_curve(y,  yproba[:,1])
auc = roc_auc_score(y, yproba[:,1])
result_table = pd.concat([result_table, pd.DataFrame([{'classifiers':"KNeighborsClassifier",
                                        'fpr':fpr, 
                                        'tpr':tpr, 
                                        'auc':auc}])], ignore_index=True)
result_table.set_index('classifiers', inplace=True)

In [81]:
fig = plt.figure(figsize=(8,6))

for i in result_table.index:
    plt.plot(result_table.loc[i]['fpr'], 
             result_table.loc[i]['tpr'], 
             label="{}, AUC={:.3f}".format(i, result_table.loc[i]['auc']))
    
plt.plot([0,1], [0,1], color='orange', linestyle='--')

plt.xticks(np.arange(0.0, 1.1, step=0.1))
plt.xlabel("Flase Positive Rate", fontsize=15)

plt.yticks(np.arange(0.0, 1.1, step=0.1))
plt.ylabel("True Positive Rate", fontsize=15)

plt.title('ROC Curve Analysis', fontweight='bold', fontsize=15)
plt.legend(prop={'size':13}, loc='lower right')

plt.show()